# AutoGluon — Predicción tasa USD/CUP (7 días)

**Objetivo:** predecir el precio promedio diario (`avg`) USD/CUP de eltoque.com
(vía API cubanomic) con AutoGluon TimeSeries, horizonte 7 días, CPU-only.

**Criterio de éxito:** el ensemble AutoGluon supera el baseline naive en MAE y
MASE sobre los últimos 30 días.

**Decisiones congeladas:** solo USD (multi-moneda listo vía `CURRENCIES`),
target `avg`, frecuencia diaria, fit completo una sola vez, re-exploración
manual (celda final).

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor

# ejecutar desde raíz del repo
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

from src.data_loader import load_series

# Config única (cambiar aquí, no en celdas sueltas)
np.random.seed(42)
PREDICTION_LENGTH = 7
TEST_DAYS = 30
FREQ = "D"
PREDICTOR_PATH = "models/predictor"
RECIPE_PATH = "models/recipe.json"
OUTPUT_PATH = "output/predictions.json"
HISTORY_PATH = "output/history.jsonl" 

In [ ]:
# Datos: fetch + fallback cache. Formato largo -> TimeSeriesDataFrame.
df_long = load_series(refresh=True)
tsdf = TimeSeriesDataFrame.from_data_frame(
    df_long, id_column="item_id", timestamp_column="timestamp"
)
print("shape:", tsdf.shape)
print(tsdf.head())
idx = tsdf.index.get_level_values("timestamp")
print("rango:", idx.min().date(), "→", idx.max().date()) 

## EDA

Serie diaria ~2 años. Análisis: tendencia (medias móviles), estacionalidad
semanal (autocorrelación lag 7) y outliers (z-score > 3).

In [ ]:
s = tsdf.loc["USD"]["target"]
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

axes[0].plot(s.index, s.values, lw=0.8)
axes[0].set_title("Serie USD/CUP (avg)")

axes[1].plot(s.index, s.rolling(7).mean(), label="MA7")
axes[1].plot(s.index, s.rolling(30).mean(), label="MA30")
axes[1].legend()
axes[1].set_title("Tendencia (medias móviles)")

z = (s - s.mean()) / s.std()
axes[2].plot(s.index, s.values, lw=0.8)
axes[2].scatter(s.index[np.abs(z) > 3], s[np.abs(z) > 3],
                color="red", s=12, label="outlier z>3")
axes[2].legend()
axes[2].set_title("Outliers (z-score > 3)")

plt.tight_layout()
plt.show()

acf = [s.autocorr(lag=k) for k in range(1, 15)]
print("outliers z>3:", int((np.abs(z) > 3).sum()))
print("autocorr lag 1..14:", [round(v, 3) for v in acf]) 

In [ ]:
# Split: últimos TEST_DAYS = test, resto = train
train = tsdf.slice_by_timestep(None, -TEST_DAYS)
test = tsdf.slice_by_timestep(-TEST_DAYS, None)
print("train:", train.shape, "| test:", test.shape) 

In [ ]:
# Baselines: naive (último valor) y media móvil 7d
s_train = train.loc["USD"]["target"]
s_test = test.loc["USD"]["target"]
s_full = tsdf.loc["USD"]["target"]

naive_pred = np.full(len(s_test), s_train.iloc[-1])
ma7_pred = np.full(len(s_test), s_train.rolling(7).mean().iloc[-1])
seasonal_pred = s_full.shift(7).reindex(s_test.index)  # naive estacional (base MASE)

def mae(y, p): return float(np.mean(np.abs(y - p)))
def rmse(y, p): return float(np.sqrt(np.mean((y - p) ** 2)))
def mase(y, p): return mae(y, p) / mae(y, seasonal_pred)

baselines = {
    "naive": {"MAE": mae(s_test, naive_pred),
              "RMSE": rmse(s_test, naive_pred),
              "MASE": mase(s_test, naive_pred)},
    "moving_avg_7": {"MAE": mae(s_test, ma7_pred),
                     "RMSE": rmse(s_test, ma7_pred),
                     "MASE": mase(s_test, ma7_pred)},
}
pd.DataFrame(baselines).round(4) 

In [ ]:
# fit() completo UNA sola vez (CPU, sin GPU).
# Excluir Chronos: muy lento en CPU.
predictor = TimeSeriesPredictor(
    prediction_length=PREDICTION_LENGTH,
    freq=FREQ,
    eval_metric="MASE",
    path=PREDICTOR_PATH,
)
predictor.fit(
    train,
    presets="medium_quality",
    time_limit=1800,              # 30 min; subir para exploración completa
    excluded_model_types=["Chronos"],
    verbosity=1,
) 

In [ ]:
# Leaderboard + análisis del ganador
leaderboard = predictor.leaderboard(test)
print(leaderboard) 

In [ ]:
# Congelar receta (config + modelos + métricas + baselines).
# Pesos del ensemble se regeneran en refit con los mismos presets (D3).
best_model = leaderboard["model"].iloc[0]
weights = {}
try:
    info = predictor.info()
    for m in info.get("model_info", {}).values():
        if isinstance(m, dict) and "weight" in m:
            weights[m.get("name", "?")] = m["weight"]
except Exception:
    weights = {}

recipe = {
    "frozen_at": pd.Timestamp.utcnow().isoformat(),
    "eval_metric": "MASE",
    "prediction_length": PREDICTION_LENGTH,
    "freq": FREQ,
    "test_days": TEST_DAYS,
    "presets": "medium_quality",
    "excluded_model_types": ["Chronos"],
    "best_model": best_model,
    "ensemble_weights": weights,
    "leaderboard": leaderboard.reset_index().to_dict(orient="records"),
    "baselines": baselines,
    "data_last_date": str(tsdf.index.get_level_values("timestamp").max().date()),
}
Path(RECIPE_PATH).parent.mkdir(parents=True, exist_ok=True)
with open(RECIPE_PATH, "w") as f:
    json.dump(recipe, f, indent=2, default=str)
print("receta congelada en", RECIPE_PATH) 

In [ ]:
# Predicción demo 7 días + plot
forecast = predictor.predict(train)
print(forecast.head(PREDICTION_LENGTH))

f = forecast.loc["USD"]
s_hist = s_full[-60:]
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(s_hist.index, s_hist.values, label="histórico", lw=1)
ax.plot(f.index, f["mean"], "o--", color="#00e5ff", label="forecast (mean)")
if "0.1" in f.columns and "0.9" in f.columns:
    ax.fill_between(f.index, f["0.1"], f["0.9"], alpha=0.25, color="#00e5ff",
                    label="p10-p90")
ax.legend()
ax.set_title("Forecast 7 días")
plt.tight_layout()
plt.show() 

## Next steps

- **Producción**: `src/predict.py` (diario), `src/retrain.py` (semanal),
  `src/monitor.py` (mensual) — leen esta receta congelada.
- **CI**: `.github/workflows/` (daily_predict, weekly_retrain, monthly_check).
- **Frontend**: `frontend/index.html` consume `output/predictions.json`.


## Re-exploración manual (D3)

Ejecutar SOLO a voluntad: cada 3-6 meses, o si `src/monitor.py` detecta
degradación (MAE_7d > 1.5× baseline). NO automatizar en CI.

Descomentar la celda siguiente y ejecutar. `time_limit` mayor para
exploración completa de hiperparámetros.

In [ ]:
# ⚠️ RE-EXPLORACIÓN MANUAL. Descomentar y ejecutar a voluntad.
# predictor2 = TimeSeriesPredictor(
#     prediction_length=PREDICTION_LENGTH, freq=FREQ,
#     eval_metric="MASE", path=PREDICTOR_PATH,
# )
# predictor2.fit(
#     tsdf, presets="best_quality", time_limit=7200,
#     excluded_model_types=["Chronos"], verbosity=2,
# )
# # luego: volver a congelar recipe.json (celda "Congelar receta")
